## Import Necessary Librires

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import warnings
import mlflow
import psutil
import time

from xgboost import XGBRFRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, BaggingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

warnings.filterwarnings('ignore')

%matplotlib inline

## Loading Data

In [81]:
df = pd.read_csv(r'../../3_Data/processed/g_2025_hourly_all_PCA_reduced.csv')
df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,taxi_demand
0,-0.518754,0.917431,-4.289254,1.130338,-0.421782,-1.711911,2.613329,-1.037458,-1.132565,1.374478,1051
1,-0.699536,0.915800,-4.206067,1.155734,-0.421137,-1.694405,2.545232,-1.031434,-1.098232,1.314072,436
2,-0.896397,0.911873,-4.015032,0.924829,-0.424154,-1.656173,2.435000,-1.014045,-1.031267,1.189971,268
3,-1.022439,0.904980,-3.811575,0.575039,-0.437192,-1.594720,2.337239,-0.991172,-0.964198,1.051892,220
4,-1.084073,0.895503,-3.635308,0.278344,-0.456338,-1.521287,2.255461,-0.967715,-0.905209,0.920323,277


In [82]:
df.shape

(6528, 11)

## **Splitig The Data**

In [83]:
X = df.drop(columns= ["taxi_demand",])
y = df['taxi_demand']

In [84]:
# Load and preprocess your data (X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [85]:
X_train.shape, X_test.shape

((5222, 10), (1306, 10))

In [86]:
y_train.shape, y_test.shape

((5222,), (1306,))

### Experiment Tracking

In [87]:
mlflow.set_experiment("Taxi_Demand_Forecasting")

<Experiment: artifact_location='file:///media/sheikh/F262ADC762AD90C1/backup/ML/yellow-taxi-demand-analysis/2_Model_Development/a_Model%20Experiment/mlruns/242402790340972245', creation_time=1763904272554, experiment_id='242402790340972245', last_update_time=1763904272554, lifecycle_stage='active', name='Taxi_Demand_Forecasting', tags={}>

### Model_1: LinearRegression

In [89]:
# For Runtime Measurement
start_time = pd.Timestamp.now()
# For Memory Usage
def get_memory_usage():
    process = psutil.Process()
    mem_info = process.memory_info().rss
    return mem_info / (1024 ** 2)  # Convert to MB

mlflow.set_experiment("Taxi_Demand_Forecasting")
with mlflow.start_run(run_name="Linear_Regression_Model"):

    lr_reg_model = LinearRegression()

    parameters = {
        "fit_intercept": [True, False],
        "copy_X": [True, False],
        "n_jobs": [None, -1],
    }

    grid_search = GridSearchCV(
        estimator=lr_reg_model,
        param_grid=parameters,
        cv=5,
        n_jobs=-1,
        scoring='neg_mean_squared_error',
    )
    # Train the model
    grid_search.fit(X_train, y_train)
    # Get the best model
    best_model = grid_search.best_estimator_
    print(f"Best Model: {best_model}")
    # Get the best params
    best_params = grid_search.best_params_
    print(f"Best Parameters: {best_params}")
    # Test the data
    y_pred = best_model.predict(X_test)
    # Log the parameters
    mlflow.log_params(grid_search.best_params_)

    # Calculate and log the evaluation metric (e.g., RMSE)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, y_pred)


    print("\n")
    print(f"RMSE: {rmse}")
    print(f"MAPE: {mape}")
    print(f"MAE: {mae}")
    print(f"MSE: {mse}")
    print(f"R2 Score: {r2}")
    print("\n")


    mlflow.log_metrics({
        "RMSE": rmse,
        "MAPE": mape,
        "MAE": mae,
        "MSE": mse,
        "R2_SCORE": r2
    })

    # Infer the model signature
    signature = infer_signature(X_train, best_model.predict(X_train))
    # Log the best model using MLflow
    mlflow.sklearn.log_model(
        best_model,
        "LinearRegression",
        input_example=X_train.iloc[:5],
        signature=signature
    )

end_time = pd.Timestamp.now()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} seconds")
print(" ",end='\n')
print(f"Memory usage: {get_memory_usage()} MB")

2025/11/23 21:59:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best Model: LinearRegression()
Best Parameters: {'copy_X': True, 'fit_intercept': True, 'n_jobs': None}


RMSE: 1318.670563379071
MAPE: 0.3272415232755023
MAE: 976.0151710750239
MSE: 1738892.0547224763
R2 Score: 0.8142175463296445


Elapsed time: 0 days 00:00:09.234706 seconds
 
Memory usage: 319.46484375 MB


## Model: All

In [ ]:
# ================= MEMORY FUNCTION =================
def get_memory_usage():
    process = psutil.Process()
    return process.memory_info().rss / (1024 ** 2)  # MB

mlflow.set_experiment("Taxi_Demand_Forecasting")

models = {
    "LinearRegression": (
        LinearRegression(),
        {"fit_intercept": [True, False]}
    ),
    "Ridge": (
        Ridge(),
        {"alpha": [0.1, 1, 10]}
    ),
    "Lasso": (
        Lasso(),
        {"alpha": [0.001, 0.01, 0.1]}
    ),
    "RandomForest": (
        RandomForestRegressor(random_state=42),
        {"n_estimators": [100, 200], "max_depth": [None, 10]}
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(random_state=42),
        {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1]}
    ),
    "KNN": (
        KNeighborsRegressor(),
        {"n_neighbors": [3, 5, 7]}
    ),
    "DecisionTree": (
        DecisionTreeRegressor(random_state=42),
        {"max_depth": [None, 10, 20]}
    ),
    "Bagging": (
        BaggingRegressor(random_state=42),
        {"n_estimators": [50, 100]}
    ),
    "XGBRF": (
        XGBRFRegressor(random_state=42),
        {"n_estimators": [100, 200], "max_depth": [6, 10]}
    ),
}

best_rmse = float("inf")
best_model_name = None
leaderboard = []

for model_name, (model, params) in models.items():

    print(f"\n Training {model_name}...")

    start_time = time.time()
    start_memory = get_memory_usage()

    with mlflow.start_run(run_name=model_name):

        grid = GridSearchCV(
            model,
            params,
            cv=5,
            scoring="neg_mean_squared_error",
            n_jobs=-1
        )

        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

        y_pred = best_model.predict(X_test)

        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)
        mape = mean_absolute_percentage_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        runtime = time.time() - start_time
        memory_used = get_memory_usage() - start_memory

        leaderboard.append([model_name, rmse, r2, mse, mae, mape, runtime, memory_used])

        print(f"""
            Model: {model_name}
            RMSE: {rmse}
            R2: {r2}
            MSE: {mse}
            MAE: {mae}
            MAPE: {mape}
            Runtime: {runtime:.2f}s
            Memory Used: {memory_used:.2f} MB
        """)

        mlflow.log_params(grid.best_params_)
        mlflow.log_metrics({
            "RMSE": rmse,
            "MSE": mse,
            "MAE": mae,
            "MAPE": mape,
            "R2": r2,
            "Runtime": runtime,
            "Memory_MB": memory_used
        })

        signature = infer_signature(X_train, best_model.predict(X_train))

        mlflow.sklearn.log_model(
            best_model,
            model_name,
            signature=signature,
            input_example=X_train.iloc[:5]
        )

        if rmse < best_rmse:
            best_rmse = rmse
            best_model_name = model_name


# ================= LEADERBOARD =================
leaderboard_df = pd.DataFrame(
    leaderboard,
    columns=["Model", "RMSE", "R2", "MSE", "MAE", "MAPE", "Runtime(s)", "Memory(MB)"]
).sort_values(by="RMSE")

print("\n MODEL LEADERBOARD (Best → Worst)")
print(leaderboard_df)

print(f"\n BEST MODEL SELECTED: {best_model_name}")



 Training LinearRegression...


2025/11/23 23:23:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



            Model: LinearRegression
            RMSE: 1318.670563379071
            R2: 0.8142175463296445
            MSE: 1738892.0547224763
            MAE: 976.0151710750239
            MAPE: 0.3272415232755023
            Runtime: 11.41s
            Memory Used: 1.68 MB
        

 Training Ridge...

            Model: Ridge
            RMSE: 1318.6868792316816
            R2: 0.8142129489450234
            MSE: 1738935.0854577916
            MAE: 976.0986305118267
            MAPE: 0.327867887202818
            Runtime: 0.16s
            Memory Used: 0.00 MB
        


2025/11/23 23:23:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



 Training Lasso...

            Model: Lasso
            RMSE: 1318.6710174692994
            R2: 0.8142174183795503
            MSE: 1738893.2523135175
            MAE: 976.0137203959748
            MAPE: 0.3272423221473342
            Runtime: 0.14s
            Memory Used: 0.00 MB
        


...


            Model: DecisionTree
            RMSE: 1085.7616024137378
            R2: 0.8740491713556475
            MSE: 1178878.2572760477
            MAE: 772.0624563721592
            MAPE: 0.2143383222194877
            Runtime: 0.89s
            Memory Used: 0.04 MB
        

 Training Bagging...

            Model: Bagging
            RMSE: 816.0809702983388
            R2: 0.9288461222755071
            MSE: 665988.1500830781
            MAE: 583.8040811638591
            MAPE: 0.17942146155643038
            Runtime: 31.99s
            Memory Used: 0.05 MB
        


2025/11/23 23:26:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



 Training XGBRF...


2025/11/23 23:27:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



            Model: XGBRF
            RMSE: 900.9612228059541
            R2: 0.9132750034332275
            MSE: 811731.125
            MAE: 656.4805908203125
            MAPE: 0.1956319361925125
            Runtime: 42.52s
            Memory Used: 0.04 MB
        

 MODEL LEADERBOARD (Best → Worst)
              Model         RMSE        R2           MSE         MAE  \
3      RandomForest   814.099199  0.929191  6.627575e+05  581.465727   
7           Bagging   816.080970  0.928846  6.659882e+05  583.804081   
4  GradientBoosting   870.630286  0.919016  7.579971e+05  646.755466   
8             XGBRF   900.961223  0.913275  8.117311e+05  656.480591   
5               KNN   988.608546  0.895581  9.773469e+05  700.006279   
6      DecisionTree  1085.761602  0.874049  1.178878e+06  772.062456   
0  LinearRegression  1318.670563  0.814218  1.738892e+06  976.015171   
2             Lasso  1318.671017  0.814217  1.738893e+06  976.013720   
1             Ridge  1318.686879  0.814213  1.7389